In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI(
    base_url=os.getenv("VLLM_BASE_URL"),
    api_key=os.getenv("VLLM_API_KEY"),
)

In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [4]:
def llm(prompt):
    response = client.responses.create(
        model=os.getenv("VLLM_MODEL"),
        input=prompt,
    )
    return response.output_text

In [6]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

It sounds like you might have meant to send this to a course instructor or administrator — I'm an AI assistant, so I don't have any information about a specific course you're referring to or the ability to enroll anyone.

That said, if you're trying to join a course late, here are some general tips:

- **Check the course website or syllabus** for enrollment deadlines or waitlist policies
- **Email the instructor directly** — many are willing to accommodate late joiners, especially early in the term
- **Contact the registrar or program coordinator** if it's a formal academic course, or the platform's support team if it's an online course
- **Look for a self-paced option** — many online courses allow enrollment at any time

If you'd like, tell me which course or platform you're asking about, and I can offer more specific suggestions.


In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(prompt)
print(answer)

Yes, you can still join the course! However, keep in mind that if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [12]:
documents[0]

{'id': '4487db3924',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I access the course modules and materials?',
 'answer': "All course materials are in the GitHub repo. Each module has its own folder (e.g. `01-overview`, `03-mcp`), and cohort-specific homework is under the `cohorts/` folder.\n\nFor the 2026 cohort, the week-by-week folders are:\n\n- Week 1: [01-ai-native-workflow](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/01-ai-native-workflow)\n- Week 2: [02-development](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/02-development)\n- Week 3: [03-deployment](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/03-deployment)\n- Week 4: [04-devops](https://github.com/DataTalksClub/ai-dev-tools-zoomcamp/tree/main/cohorts/2026/04-devops)\n\nStart from the folder for your current week — it gathers everything for that week in one place.\n\n

In [13]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [14]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [15]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

In [16]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [17]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [18]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Course: How do I start?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?']

In [19]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [20]:
search_results = search(question)

In [21]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [22]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [23]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [24]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [25]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [27]:
response = client.responses.create(
        model=os.getenv("VLLM_MODEL"),
        input=prompt,
    )

In [28]:
response.output[0]

ResponseReasoningItem(id='rs_8ef21454cc4c1a34', summary=[], type='reasoning', content=[Content(text='The user is asking a simple question: "I just discovered the course. Can I join now?"\n\nBased on the context provided (FAQ from what appears to be an LLM Zoomcamp course), I should answer based on the relevant FAQ entries:\n\n1. "Q: I just discovered the course. Can I still join? A: Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions."\n\n2. Also relevant: "You can also just start learning and submitting homework (while the form is open) without registering."\n\n3. "You can work through the material and prepare your project in self-paced mode, but project submission and peer review must happen while a live cohort is accepting them."\n\nSo the answer is: Yes, you can join. You can start learning right away. However, if you want a certificate, you need to finish the capstone project and complete peer reviews while a live

In [29]:
response.output[0].content[0].text

'The user is asking a simple question: "I just discovered the course. Can I join now?"\n\nBased on the context provided (FAQ from what appears to be an LLM Zoomcamp course), I should answer based on the relevant FAQ entries:\n\n1. "Q: I just discovered the course. Can I still join? A: Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions."\n\n2. Also relevant: "You can also just start learning and submitting homework (while the form is open) without registering."\n\n3. "You can work through the material and prepare your project in self-paced mode, but project submission and peer review must happen while a live cohort is accepting them."\n\nSo the answer is: Yes, you can join. You can start learning right away. However, if you want a certificate, you need to finish the capstone project and complete peer reviews while a live cohort is still accepting submissions. Registration isn\'t strictly necessary — it\'s just to gauge

In [30]:
response.output_text

"Yes, you can join! You're free to start learning right away — no formal registration is required (registration is mainly just to gauge interest before the start date, so you don't need to wait for a confirmation email).\n\nThe main thing to keep in mind is the certificate: you can only earn one by finishing the capstone project and completing the required peer reviews **while a live cohort is still accepting submissions**. You can work through the material at your own pace, but if a certificate is your goal, make sure your project submission happens during an active cohort's submission window."

In [31]:
response.usage

ResponseUsage(input_tokens=516, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0, input_tokens_per_turn=[], cached_tokens_per_turn=[]), output_tokens=394, output_tokens_details=OutputTokensDetails(reasoning_tokens=275, tool_output_tokens=0, output_tokens_per_turn=[], tool_output_tokens_per_turn=[]), total_tokens=910)

In [32]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00216

In [33]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = client.responses.create(
    model=os.getenv("VLLM_MODEL"),
    input=message_history
)

In [34]:
def llm(instructions, user_prompt, model=os.getenv("VLLM_MODEL")):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [35]:
def rag(query, model=os.getenv("VLLM_MODEL")):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [36]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join the course even if you've just discovered it. However, there are a few things to keep in mind:

1. **Joining**: You can start learning and submitting homework right away. Registration is just to gauge interest before the start date — you don't need a confirmation email, as you're accepted automatically.

2. **Certificate requirements**: If you want to receive a certificate, you'll need to:
   - Submit your capstone project while a live cohort is still accepting submissions
   - Complete the required peer reviews

   Note that you can only get a certificate if you finish the course with a "live" cohort. You can work through the material in self-paced mode, but project submission and peer review must happen while a live cohort is accepting them.
